<a href="https://colab.research.google.com/github/skumarrms/lakebridge/blob/lakebrdige-code/Lakebridge_Databricks_to_Databrikcs_Reconcile_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Databricks notebook

dbutils.widgets.text("source_catalog", "")
dbutils.widgets.text("source_schema", "")
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
#dbutils.widgets.text("report_type", "")
dbutils.widgets.dropdown(
    "report_type",
    "row",
    ["row", "data", "schema", "all"]
)
dbutils.widgets.text("agg_column", "")
dbutils.widgets.text("agg_type", "")
dbutils.widgets.text("join_columns", "")
dbutils.widgets.text("source_columns", "")
dbutils.widgets.text("target_columns", "")



# Get params
source_catalog = dbutils.widgets.get("source_catalog")
source_schema = dbutils.widgets.get("source_schema")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
report_type = dbutils.widgets.get("report_type")
agg_column = dbutils.widgets.get("agg_column")
agg_type = dbutils.widgets.get("agg_type")
join_columns = dbutils.widgets.get("join_columns").split(",")
source_columns = dbutils.widgets.get("source_columns").split(",")
target_columns = dbutils.widgets.get("target_columns").split(",")



In [ ]:
#%sql
#drop table source_sf_catalog.source_sf_schema.product_prod;

In [ ]:
#%sql
#SHOW CATALOGS;

In [ ]:
"""
%sql
-- Step 1: Create the catalog
CREATE CATALOG IF NOT EXISTS lakebridge_metadata;

-- Step 2: Create a schema (if required)
CREATE SCHEMA IF NOT EXISTS lakebridge_metadata.reconcile;

-- Step 3: Create a volume (UC-managed storage location)
CREATE VOLUME IF NOT EXISTS lakebridge_metadata.reconcile.reconcile_volume;"""

In [ ]:

%sql
CREATE SCHEMA IF NOT EXISTS source_sf_catalog.source_sf_schema;
drop table source_sf_catalog.source_sf_schema.product_prod;
CREATE TABLE IF NOT EXISTS source_sf_catalog.source_sf_schema.product_prod (
    p_id INT,
    p_name STRING,
    price DOUBLE,
    discount DECIMAL(5,3),
    offer DOUBLE,
    creation_date DATE,
    comment STRING
);


INSERT INTO source_sf_catalog.source_sf_schema.product_prod VALUES
(1, 'Laptop',     75000, 0.075, 5.1,  DATE '2024-07-01', 'Back-to-school offer'),
(2, 'Tablet',     30000, 0.050, 2.2,  DATE '2024-06-15', 'Limited stock'),
(3, 'Smartphone', 45000, 0.100, 3.1,  DATE '2024-07-10', 'New launch'),
(4, 'Monitor',    20000, 0.025, 1,  DATE '2024-05-30', 'Year-end clearance'),
(5, 'Keyboard',    2500, 0.010, 0.5,  DATE '2024-04-20', 'Regular item');


num_affected_rows,num_inserted_rows
5,5


In [ ]:
%sql
select * from source_sf_catalog.source_sf_schema.product_prod;

p_id,p_name,price,discount,offer,creation_date,comment
1,Laptop,75000.0,0.075,5.1,2024-07-01,Back-to-school offer
2,Tablet,30000.0,0.050,2.2,2024-06-15,Limited stock
3,Smartphone,45000.0,0.100,3.1,2024-07-10,New launch
4,Monitor,20000.0,0.025,1.0,2024-05-30,Year-end clearance
5,Keyboard,2500.0,0.010,0.5,2024-04-20,Regular item


In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS target_catalog.target_databricks_schema;
drop table target_catalog.target_databricks_schema.product;
CREATE TABLE IF NOT EXISTS target_catalog.target_databricks_schema.product (
    product_id INT,
    product_name STRING,
    price STRING,
    discount DECIMAL(5,3),
    offer DOUBLE,
    creation_date DATE,
    comment STRING
);

INSERT INTO target_catalog.target_databricks_schema.product  VALUES
(1, 'Laptop',     75000, 0.075, 5.5,  DATE '2024-07-01', 'Back-to-school offer'),
(2, 'Tablet',     30000, 0.050, 2.4,  DATE '2024-06-15', 'Limited stock'),
(3, 'Smartphone', 45000, 0.100, 3.6,  DATE '2024-07-10', 'New launch'),
(4, 'Monitor',    20000, 0.025, 1,  DATE '2024-05-30', 'Year-end clearance'),
(5, 'Keyboard',    2500, 0.010, 0.5,  DATE '2024-04-20', 'Regular item');

num_affected_rows,num_inserted_rows
5,5


In [ ]:
%sql
select * from target_catalog.target_databricks_schema.product;

product_id,product_name,price,discount,offer,creation_date,comment
1,Laptop,75000,0.075,5.5,2024-07-01,Back-to-school offer
2,Tablet,30000,0.050,2.4,2024-06-15,Limited stock
3,Smartphone,45000,0.100,3.6,2024-07-10,New launch
4,Monitor,20000,0.025,1.0,2024-05-30,Year-end clearance
5,Keyboard,2500,0.010,0.5,2024-04-20,Regular item


In [ ]:
%pip install git+https://github.com/databrickslabs/lakebridge
dbutils.library.restartPython()

  Cloning https://github.com/databrickslabs/lakebridge to /tmp/pip-req-build-_tq2ywtx
  Running command git clone --filter=blob:none --quiet https://github.com/databrickslabs/lakebridge /tmp/pip-req-build-_tq2ywtx
  Resolved https://github.com/databrickslabs/lakebridge to commit b647d0e9b7ca5de9fa80d2b4e3b5b4ce976c94fd
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig,
    TableRecon
)
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)
from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.labs.lakebridge.reconcile.trigger_recon_aggregate_service import TriggerReconAggregateService
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

In [ ]:
from dataclasses import dataclass

@dataclass
class ReconcileConfig:
    data_source: str
    report_type: str
    secret_scope: str
    database_config: DatabaseConfig
    metadata_config: ReconcileMetadataConfig

In [ ]:
@dataclass
class DatabaseConfig:
    source_schema: str
    target_catalog: str
    target_schema: str
    source_catalog: str | None = None

In [ ]:
@dataclass
class ReconcileMetadataConfig:
    catalog: str = "lakebridge"
    schema: str = "reconcile"
    volume: str = "reconcile_volume"

In [ ]:
@dataclass
class Aggregate:
    agg_columns: list(agg_column)
    type: agg_type
    group_by_columns: list[str] | None = None

In [ ]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)

reconcile_config = ReconcileConfig(
    data_source="databricks",
    report_type=report_type,
    secret_scope="lakebridge_data_source",  # Optional if authentication is already handled
    database_config=DatabaseConfig(
        source_catalog=source_catalog,  # or your real catalog, if used
        source_schema=source_schema,
        target_catalog=target_catalog,  # or your real catalog, if used
        target_schema=target_schema
    ),
    metadata_config=ReconcileMetadataConfig(
        catalog="lakebridge_metadata",
        schema="reconcile"
    )
)

In [ ]:
@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

In [ ]:
from databricks.labs.lakebridge.config import TableRecon
from databricks.labs.lakebridge.reconcile.recon_config import (
    Table,
    ColumnMapping,
    ColumnThresholds,
    TableThresholds,
    Transformation,
    JdbcReaderOptions,
    Aggregate,
    Filters
)

table_recon = TableRecon(
    source_catalog="source_catalog",
    source_schema="source_schema",
    target_catalog="target_catalog",
    target_schema="target_schema",
    tables=[
        Table(
            #source_name="product_prod",
            #target_name="product",
            source_name=source_table,
            target_name=target_table,
            join_columns=join_columns,
            drop_columns=["comment"],
            #column_mapping=[
            #    ColumnMapping(source_name="p_id", target_name="product_id"),
            #    ColumnMapping(source_name="p_name", target_name="product_name")
            #],
            column_mapping = [
                ColumnMapping(source_name=s, target_name=t)
                for s, t in zip(source_columns, target_columns)
            ],
            transformations=[
                Transformation(
                    column_name="creation_date",
                    source="creation_date",
                    target="to_date(creation_date,'yyyy-mm-dd')"
                )
            ],
            ##############
            aggregates=[
                Aggregate(
                    agg_columns=agg_column,
                    type=agg_type,
                    #group_by_columns=[]
                )
            ],
            #############
            column_thresholds=[
                ColumnThresholds(column_name="price", upper_bound="-50", lower_bound="50", type="float")
            ],
            table_thresholds=[
                TableThresholds(lower_bound="0%", upper_bound="5%", model="mismatch")
            ],
            jdbc_reader_options=JdbcReaderOptions(
                number_partitions=10,
                partition_column="p_id",
                lower_bound="0",
                upper_bound="10000000"
            ),
            filters=Filters(
                source="p_id > 0",
                target="product_id > 0"
            )
        )
    ]
)


In [ ]:
from databricks.labs.lakebridge import __version__
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

ws = WorkspaceClient(product="lakebridge", product_version=__version__)

try:
  result = TriggerReconService.trigger_recon(
            ws = ws,
            spark = spark, # notebook spark session
            table_recon = table_recon, # previously created
            reconcile_config = reconcile_config # previously created
          )
  print(result.recon_id)
  print(result)
  print("***************************")
except ReconciliationException as e:
    recon_id = e.reconcile_output.recon_id
    print(f" Failed : {recon_id}")

    print(e)
    print("***************************")
except Exception as e:
    print(e.with_traceback)
    raise e
    print(f"Exception : {str(e)}")
    print("***************************")


12:03:31  WARNING [d.l.l.reconcile.compare] Unmatched data was written to /Volumes/lakebridge_metadata/reconcile/reconcile_volume/product_prod_product/ successfully
12:03:32  WARNING [d.l.l.reconcile.reconciliation] Threshold comparison is ignored for 'row' report type
12:03:32  WARNING [d.l.l.reconcile.trigger_recon_service] Reconciliation for 'row' report completed.
12:03:35  WARNING [d.l.l.reconcile.recon_capture] reconciled_record_count : ReconcileRecordCount(source=5, target=5)
12:03:42  WARNING [d.l.l.reconcile.recon_capture] Unmatched DF cleaned up from /Volumes/lakebridge_metadata/reconcile/reconcile_volume/product_prod_product/ successfully.


 Failed : 703361e1-157d-4406-a102-14ee80b41d32
(' Reconciliation failed for one or more tables. Please check the recon metrics for more details. **reconcile** failed.', ReconcileOutput(recon_id='703361e1-157d-4406-a102-14ee80b41d32', results=[ReconcileTableOutput(target_table_name='target_catalog.target_databricks_schema.product', source_table_name='source_sf_catalog.source_sf_schema.product_prod', status=StatusOutput(row=False, column=None, schema=None, aggregate=None), exception_message='')]))
***************************


In [ ]:

from databricks.labs.lakebridge import __version__
from databricks.sdk import WorkspaceClient

from databricks.labs.lakebridge.reconcile.execute.reconcile_aggregates import reconcile_aggregates
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException

# Initialize Workspace Client
ws = WorkspaceClient(product="lakebridge", product_version=__version__)

try:
    # Run aggregate reconciliation
    result = reconcile_aggregates(
        ws=ws,
        spark=spark,  # Use current Spark session
        table_recon=table_recon,  # Existing table reconciliation object
        reconcile_config=reconcile_config  # Existing config with aggregate section
    )

    # Print results
    print(result.recon_id)
    print(result)
    print("***************************")

except ReconciliationException as e:
    recon_id = e.reconcile_output.recon_id
    print(f"❌ Reconciliation Failed: {recon_id}")
    print(e)
    print("***************************")

except Exception as e:
    print("❌ Unexpected Error:")
    print(e)
    print("***************************")
    raise e


---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-6591593587925353>, line 4
      1 from databricks.labs.lakebridge import __version__
      2 from databricks.sdk import WorkspaceClient
----> 4 from databricks.labs.lakebridge.reconcile.execute import reconcile_aggregates
      5 from databricks.labs.lakebridge.reconcile.exception import ReconciliationException
      7 # Initialize Workspace Client

ImportError: cannot import name 'reconcile_aggregates' from 'databricks.labs.lakebridge.reconcile.execute' (/local_disk0/.ephemeral_nfs/envs/pythonEnv-5a9b2e41-19d6-4141-9939-951cb1bd9872/lib/python3.11/site-packages/databricks/labs/lakebridge/reconcile/execute.py)

In [ ]:
#retain_unmatched_df = spark.read.format("parquet").load("/Volumes/lakebridge_metadata/reconcile/reconcile_volume/product_prod_product/")
#display(retain_unmatched_df)


In [ ]:
%sql
select * from lakebridge_metadata.reconcile.main where recon_id = 'f2199ee0-6ee0-48af-99ae-256f9b9e6538';

recon_table_id,recon_id,source_type,source_table,target_table,report_type,operation_name,start_ts,end_ts
6236946608461154956,f2199ee0-6ee0-48af-99ae-256f9b9e6538,Databricks,"List(source_sf_catalog, source_sf_schema, product_prod)","List(target_catalog, target_databricks_schema, product)",row,reconcile,2025-09-15T11:02:18.286Z,2025-09-15T11:02:24.221Z


In [ ]:
%sql
select * from lakebridge_metadata.reconcile.details where recon_table_id = 6236946608461154956;

recon_table_id,recon_type,status,data,inserted_ts
6236946608461154956,missing_in_source,false,"List(Map(p_name -> Tablet, p_id -> 2, creation_date -> 2024-06-15, offer -> 2.4, discount -> 0.050), Map(p_name -> Smartphone, p_id -> 3, creation_date -> 2024-07-10, offer -> 3.6, discount -> 0.100), Map(p_name -> Laptop, p_id -> 1, creation_date -> 2024-07-01, offer -> 5.5, discount -> 0.075))",2025-09-15T11:02:29.498Z
6236946608461154956,missing_in_target,false,"List(Map(p_name -> Smartphone, p_id -> 3, creation_date -> 2024-07-10, offer -> 3.1, discount -> 0.100), Map(p_name -> Laptop, p_id -> 1, creation_date -> 2024-07-01, offer -> 5.1, discount -> 0.075), Map(p_name -> Tablet, p_id -> 2, creation_date -> 2024-06-15, offer -> 2.2, discount -> 0.050))",2025-09-15T11:02:31.638Z


In [ ]:
%sql
select * from lakebridge_metadata.reconcile.metrics where recon_table_id = 6236946608461154956;

recon_table_id,recon_metrics,run_metrics,inserted_ts
6236946608461154956,"List(List(3, 3), null, null)","List(false, kaledhiraj7777@gmail.com, )",2025-09-15T11:02:27.340Z


In [ ]:
"""
from importlib.resources import files
from pathlib import Path

import databricks.labs.lakebridge.resources
from databricks.labs.lakebridge import __version__
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig,
)
from databricks.labs.lakebridge.contexts.application import ApplicationContext
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient(product="lakebridge", product_version=__version__)
app_context = ApplicationContext(ws)
resources = files(databricks.labs.lakebridge.resources)
dashboards_dir = resources.joinpath("reconcile/dashboards")
if not isinstance(dashboards_dir, Path):
    raise TypeError("Dashboards directory is not a real filesystem path.")

# Create a ReconcileConfig instance with the necessary configurations.
# Make sure the objects exist.
recon_config = ReconcileConfig(
    data_source="databricks",
    report_type="row",
    secret_scope="lakebridge_data_source",
    database_config=DatabaseConfig(
        source_schema="source_sf_schema",
        target_catalog="target_catalog",
        target_schema="target_databricks_schema",
    ),
    metadata_config=ReconcileMetadataConfig(
        catalog="lakebridge_metadata",
        schema="reconcile",
        volume="reconcile_volume",
    ),
)

query_dir = resources.joinpath("reconcile/queries/installation")
sqls_to_deploy = [
    "main.sql",
    "metrics.sql",
    "details.sql",
    "aggregate_metrics.sql",
    "aggregate_details.sql",
    "aggregate_rules.sql",
]
for sql_file in sqls_to_deploy:
    table_sql_file = query_dir.joinpath(sql_file)
    app_context.table_deployment.deploy_table_from_ddl_file(
        recon_config.metadata_config.catalog,
        recon_config.metadata_config.schema,
        sql_file.strip(".sql"),
        table_sql_file,
    )

app_context.dashboard_deployment.deploy(folder=Path(dashboards_dir), config=recon_config)
app_context.install_state.save()
"""

'\nfrom importlib.resources import files\nfrom pathlib import Path\n\nimport databricks.labs.lakebridge.resources\nfrom databricks.labs.lakebridge import __version__\nfrom databricks.labs.lakebridge.config import (\n    DatabaseConfig,\n    ReconcileConfig,\n    ReconcileMetadataConfig,\n)\nfrom databricks.labs.lakebridge.contexts.application import ApplicationContext\nfrom databricks.sdk import WorkspaceClient\n\nws = WorkspaceClient(product="lakebridge", product_version=__version__)\napp_context = ApplicationContext(ws)\nresources = files(databricks.labs.lakebridge.resources)\ndashboards_dir = resources.joinpath("reconcile/dashboards")\nif not isinstance(dashboards_dir, Path):\n    raise TypeError("Dashboards directory is not a real filesystem path.")\n\n# Create a ReconcileConfig instance with the necessary configurations.\n# Make sure the objects exist.\nrecon_config = ReconcileConfig(\n    data_source="databricks",\n    report_type="row",\n    secret_scope="lakebridge_data_source"